# Selection of largest lakes
This is a small script to check which lakes are smartest to use for the Sentinel-3 analysis. The issue is that S3 has a very coarse resolution of 1km, and thus there has to be not only a large lake surface, but also a good shape. For example, long and very narrow lakes can have a large area, but no 1kmx1km pixels within the lake shorelines. Thus, I check how many 1x1km pixels fit into the 100 largest ice-marginal lakes in this script. 

In [2]:
# import modules
import pandas as pd
import geopandas as gpd
import numpy as np
import shapely as shp
import xarray as xr
import rasterio
import matplotlib.pyplot as plt
import cartopy

In [3]:
# check the largest lakes by area, when subtracting 300m buffer and then how many pixels in square kilometer
iml = gpd.read_file(r"..\lake_polygons\ALL-ESA-GRIML-IML-fv4.gpkg")


In [ ]:
# sort for the largest lakes and then just buffer for the largest 500
iml_sorted = iml.sort_values(by = ['area_2025'], ascending=False)
iml_500 = iml_sorted[:100] # get first 100 by area to make pixel calculation faster (4000 otherwise)

In [31]:
# -500 m buffer for the edge effects
# takes a long time to run... someting like 10min
iml_buffered = iml_500.copy()
iml_buffered["geometry"] = iml_500.geometry.buffer(-300)

In [32]:
# check the workflow with one lake and known number of final pixels: must be something like 78/80
lake6 = gpd.read_file(r'..\lake_polygons\lake_6_shoreline_shp\lake_shoreline.shp')
lake6 = lake6.to_crs(epsg = 3413) # reproject for unit = m (otherwise in degrees)
lake6_buffered = lake6.buffer(-300)

In [33]:
# create grid
import geopandas as gpd
import numpy as np
from shapely.geometry import box

def grid_over_geometry(geometry, cell_size=1000):
    minx, miny, maxx, maxy = geometry.bounds

    xs = np.arange(minx, maxx + cell_size, cell_size)
    ys = np.arange(miny, maxy + cell_size, cell_size)

    cells = [
        box(x, y, x + cell_size, y + cell_size)
        for x in xs[:-1]
        for y in ys[:-1]
    ]

    return gpd.GeoDataFrame(geometry=cells, crs="EPSG:3413")

lake6_grid = grid_over_geometry(lake6_buffered.geometry.iloc[0], 1000)
inside = lake6_grid[lake6_grid.geometry.within(lake6_buffered.geometry.iloc[0])]
inside

,geometry
13,"POLYGON ((-250590.646 -3023759.906, -250590.64..."
21,"POLYGON ((-249590.646 -3024759.906, -249590.64..."
30,"POLYGON ((-248590.646 -3024759.906, -248590.64..."
39,"POLYGON ((-247590.646 -3024759.906, -247590.64..."
48,"POLYGON ((-246590.646 -3024759.906, -246590.64..."
57,"POLYGON ((-245590.646 -3024759.906, -245590.64..."
66,"POLYGON ((-244590.646 -3024759.906, -244590.64..."
67,"POLYGON ((-244590.646 -3023759.906, -244590.64..."
75,"POLYGON ((-243590.646 -3024759.906, -243590.64..."
76,"POLYGON ((-243590.646 -3023759.906, -243590.64..."


In [34]:
import pandas as pd

results = []

for idx, row in iml_buffered.iterrows():

    lake_id = row["lake_id"]
    geom = row.geometry

    # Empty geometry after inward buffer
    if geom.is_empty:
        num_pixels = 0

    else:
        # Create 1 km grid
        grid = grid_over_geometry(geom, cell_size=1000)

        # Pixels completely inside
        inside = grid.geometry.within(geom)

        num_pixels = inside.sum()

    results.append({
        "lake_id": lake_id,
        "num_pixels": num_pixels
    })

pixel_counts = pd.DataFrame(results)

In [35]:
# get the lakes with most number of pixels
pixel_counts[pixel_counts.num_pixels>0].sort_values(by = ['num_pixels'], ascending=False)[0:15]
largest = pixel_counts[pixel_counts.num_pixels>0].sort_values(by = ['num_pixels'], ascending=False)[0:10].lake_id.values

In [36]:
iml.merge(pixel_counts, how = "inner", on = "lake_id").sort_values(by = ['num_pixels'], ascending=False)[0:10]

,row_id,lake_id,lake_name,margin,region,area_all,area_2016,area_2017,area_2018,area_2019,...,startdate,enddate,method,source,verified,verif_by,edited,edited_by,geometry,num_pixels
1,2,2,Romer Sø,ICE_CAP,NE,128.895428,123.613100,121.7751,15.131300,105.097500,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((426330 -886400, 426320 -886400...",66
0,1,1,Unknown,ICE_SHEET,NO,131.663559,10.744700,NaN,NaN,11.340100,...,20160701,20250831,Digitisation,Klimadatastyrelsen Open Land Greenland vector ...,yes,How,NaN,NaN,"MULTIPOLYGON (((-30010 -920704.644, -29998.709...",37
3,4,4,Unknown,ICE_SHEET,SW,105.220790,82.579500,NaN,40.275500,72.155500,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,NaN,"MULTIPOLYGON (((-244060.935 -2627241.449, -244...",37
5,6,6,Kangaarsuup Tasersua,ICE_SHEET,SW,84.101255,81.100800,81.3906,70.875200,81.161500,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((-251920 -3022420, -251920 -302...",23
2,3,3,Sælsøen,ICE_SHEET,NE,111.376216,30.335100,NaN,102.103322,104.034658,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((553420 -1294210, 553410 -12942...",21
6,7,7,Unknown,ICE_SHEET,NE,75.719700,65.857500,NaN,63.329900,68.649500,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((437760 -1050600, 437760 -10506...",18
7,8,8,"Ilulialik (AP), Isortuarsuup Tasia",ICE_SHEET,SW,64.285046,62.119100,62.5718,NaN,61.709100,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((-254850 -2869630, -254850 -286...",16
11,12,12,Inuiteq Sø,ICE_SHEET,NO,50.499536,49.130098,NaN,45.895800,48.558796,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((148720 -865780, 148720 -865790...",13
8,9,9,Furesø,ICE_CAP,NE,55.541885,53.238065,NaN,NaN,44.311351,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((633510 -1859166.924, 633507.71...",10
13,16,16,Tasersiaq,ICE_SHEET,SW,45.053546,8.469700,NaN,5.743000,1.222500,...,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((-223030 -2592760, -223030 -259...",8


In [37]:
iml_largest = iml[iml["lake_id"].isin(largest)]
iml_largest

,row_id,lake_id,lake_name,margin,region,area_all,area_2016,area_2017,area_2018,area_2019,...,t_date_2025,startdate,enddate,method,source,verified,verif_by,edited,edited_by,geometry
0,1,1,Unknown,ICE_SHEET,NO,131.663559,10.744700,NaN,NaN,11.340100,...,"2025-08-11 17:07, 2025-08-19 17:07, 2025-08-11...",20160701,20250831,Digitisation,Klimadatastyrelsen Open Land Greenland vector ...,yes,How,NaN,NaN,"MULTIPOLYGON (((-30010 -920704.644, -29998.709..."
1,2,2,Romer Sø,ICE_CAP,NE,128.895428,123.613100,121.7751,15.131300,105.097500,...,"2025-08-11 18:44, 2025-08-11 17:06, 2025-08-19...",20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((426330 -886400, 426320 -886400..."
2,3,3,Sælsøen,ICE_SHEET,NE,111.376216,30.335100,NaN,102.103322,104.034658,...,"2025-08-28 12:56, 2025-08-19 13:02",20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((553420 -1294210, 553410 -12942..."
3,4,4,Unknown,ICE_SHEET,SW,105.220790,82.579500,NaN,40.275500,72.155500,...,2025-08-28 13:49,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,NaN,"MULTIPOLYGON (((-244060.935 -2627241.449, -244..."
5,6,6,Kangaarsuup Tasersua,ICE_SHEET,SW,84.101255,81.100800,81.3906,70.875200,81.161500,...,NaN,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((-251920 -3022420, -251920 -302..."
6,7,7,Unknown,ICE_SHEET,NE,75.719700,65.857500,NaN,63.329900,68.649500,...,"2025-08-19 14:39, 2025-08-19 14:40, 2025-08-19...",20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((437760 -1050600, 437760 -10506..."
7,8,8,"Ilulialik (AP), Isortuarsuup Tasia",ICE_SHEET,SW,64.285046,62.119100,62.5718,NaN,61.709100,...,NaN,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((-254850 -2869630, -254850 -286..."
8,9,9,Furesø,ICE_CAP,NE,55.541885,53.238065,NaN,NaN,44.311351,...,NaN,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((633510 -1859166.924, 633507.71..."
11,12,12,Inuiteq Sø,ICE_SHEET,NO,50.499536,49.130098,NaN,45.895800,48.558796,...,"2025-08-19 17:07, 2025-08-19 17:07, 2025-08-11...",20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,Yes,How,"MULTIPOLYGON (((148720 -865780, 148720 -865790..."
15,16,16,Tasersiaq,ICE_SHEET,SW,45.053546,8.469700,NaN,5.743000,1.222500,...,2025-08-28 13:49,20160701,20250831,Automatic,"Sentinel-1, Sentinel-2",yes,How,NaN,NaN,"MULTIPOLYGON (((-223030 -2592760, -223030 -259..."


In [38]:
iml_largest.to_file(r"C:\Users\Lenovo\Documents\GEUS_internship\LWST_estimation_project\lake_polygons\largest_lakes_num_pixels.geojson")
print(f"Done!")

Done!
